# How to read the filtered exp_ASR data

This notebook demonstrates how to access and read the filtered exp_ASR data.

It loads the full `EGshelfIIseas2km_ASR_full` OceanSpy dataset from the catalog. If requested, it replaces the original `U`, `V`, `W`, `S`, and `Temp` fields with the corresponding 20-day-lowpass-filtered variables stored as Zarr files.

No filtering is performed here. The notebook only opens the existing dataset and organizes the pre-filtered variables into an `OceanDataset`.

Yilang Xu,TWNH, June '26

## 1. Import packages

The notebook uses OceanSpy to open the catalog entry, xarray to open Zarr files, and standard Python utilities to locate the filtered files.

In [1]:
import glob
import os

import oceanspy as ospy
import xarray as xr

## 2. Set data-reading options

Set `use_filtered_data = True` to replace the original variables with the 20-day-lowpass-filtered versions. Set it to `False` to read the unfiltered catalog dataset only.

In [2]:
# Choose whether to use the 20-day-lowpass-filtered variables.
use_filtered_data = True

# Location of the filtered Zarr files.
filtered_data_root = "/home/idies/workspace/ocean_circulation_ceph/exp_ASR_filtered_zarr"

# Variables that have filtered Zarr versions.
filtered_var_names = ["U", "V", "W", "S", "Temp"]

## 3. Open the base OceanSpy dataset

This opens the full `exp_ASR` dataset from the OceanSpy catalog. The object `od` is the OceanSpy `OceanDataset`, and `od.dataset` gives the underlying xarray dataset.

In [3]:
od = ospy.open_oceandataset.from_catalog("EGshelfIIseas2km_ASR_full")

# Underlying xarray dataset before replacing any variables.
ds_base = od.dataset
ds_base

Opening EGshelfIIseas2km_ASR_full.
High-resolution (~2km) numerical simulation covering the east Greenland shelf (EGshelf),
and the Iceland and Irminger Seas (IIseas) forced by the Arctic System Reanalysis (ASR).
Citation:
 * Almansi et al., 2020 - GRL.
Characteristics:
 * full: Full domain without variables to close budgets.
See also:
 * EGshelfIIseas2km_ASR_crop: Cropped domain with variables to close budgets.


<xarray.Dataset> Size: 18TB
Dimensions:    (Y: 880, X: 960, time: 1464, Z: 216, Yp1: 881, Xp1: 961,
                Zp1: 217, Zl: 216, Zu: 216, time_midp: 1463)
Coordinates: (12/18)
  * Y          (Y) float64 7kB 56.81 56.85 56.89 56.93 ... 76.41 76.44 76.48
  * X          (X) float64 8kB -46.92 -46.83 -46.74 -46.65 ... 1.156 1.244 1.332
    XC         (Y, X) float64 7MB dask.array<chunksize=(220, 240), meta=np.ndarray>
    XV         (Yp1, X) float64 7MB dask.array<chunksize=(221, 240), meta=np.ndarray>
    YC         (Y, X) float64 7MB dask.array<chunksize=(220, 240), meta=np.ndarray>
    YV         (Yp1, X) float64 7MB dask.array<chunksize=(221, 240), meta=np.ndarray>
    ...         ...
    YG         (Yp1, Xp1) float64 7MB dask.array<chunksize=(221, 241), meta=np.ndarray>
    YU         (Y, Xp1) float64 7MB dask.array<chunksize=(220, 241), meta=np.ndarray>
  * Zp1        (Zp1) float64 2kB 0.0 -2.0 -5.0 ... -3.134e+03 -3.149e+03
  * Zl         (Zl) float64 2kB 0.0 -2.0 -5.0 ... -3.119e+03 -3.134e+03
  * Zu         (Zu) float64 2kB -2.0 -5.0 -9.0 ... -3.134e+03 -3.149e+03
  * time_midp  (time_midp) datetime64[ns] 12kB 2007-09-01T03:00:00 ... 2008-0...
Data variables: (12/75)
    Depth      (Y, X) float64 7MB dask.array<chunksize=(220, 240), meta=np.ndarray>
    EXFaqh     (time, Y, X) float64 10GB dask.array<chunksize=(2, 220, 240), meta=np.ndarray>
    EXFatemp   (time, Y, X) float64 10GB dask.array<chunksize=(2, 220, 240), meta=np.ndarray>
    EXFempmr   (time, Y, X) float64 10GB dask.array<chunksize=(2, 220, 240), meta=np.ndarray>
    EXFevap    (time, Y, X) float64 10GB dask.array<chunksize=(2, 220, 240), meta=np.ndarray>
    EXFhl      (time, Y, X) float64 10GB dask.array<chunksize=(2, 220, 240), meta=np.ndarray>
    ...         ...
    rA         (Y, X) float64 7MB dask.array<chunksize=(220, 240), meta=np.ndarray>
    rAs        (Yp1, X) float64 7MB dask.array<chunksize=(221, 240), meta=np.ndarray>
    rAw        (Y, Xp1) float64 7MB dask.array<chunksize=(220, 241), meta=np.ndarray>
    rAz        (Yp1, Xp1) float64 7MB dask.array<chunksize=(221, 241), meta=np.ndarray>
    surForcS   (time, Y, X) float64 10GB dask.array<chunksize=(2, 220, 240), meta=np.ndarray>
    surForcT   (time, Y, X) float64 10GB dask.array<chunksize=(2, 220, 240), meta=np.ndarray>
Attributes:
    original_attrs:        {'MITgcm_URL': 'http://mitgcm.org', 'MITgcm_mnc_ve...
    OceanSpy_parameters:   {'rSphere': 6371.0, 'eq_state': 'jmd95', 'rho0': 1...
    OceanSpy_name:         EGshelfIIseas2km_ASR_full
    OceanSpy_description:  High-resolution (~2km) numerical simulation coveri...
    OceanSpy_projection:   Mercator(**{})
    OceanSpy_grid_coords:  {'Y': {'Y': None, 'Yp1': 0.5}, 'X': {'X': None, 'X...

## 4. Define a helper function for one filtered variable

Each filtered variable is stored as a sequence of Zarr files, one file per time block. The helper function opens all matching files, concatenates them along `time`, assigns the catalog time coordinate, and renames the filtered field back to the original variable name.

In [4]:
def open_filtered_variable(var_name, filtered_data_root, time_coord):
    """Open and concatenate the filtered Zarr files for one variable."""
    zarr_pattern = os.path.join(
        filtered_data_root,
        f"{var_name}_filt_each_time",
        f"{var_name}_filt_lowpass_20_time_*.zarr",
    )
    zarr_files = sorted(glob.glob(zarr_pattern))

    if not zarr_files:
        raise FileNotFoundError(
            f"No filtered Zarr files found for {var_name}: {zarr_pattern}"
        )

    filtered_datasets = [xr.open_zarr(zarr_file) for zarr_file in zarr_files]
    filtered_data = xr.concat(filtered_datasets, dim="time")[
        f"{var_name}_filt_lowpass_20"
    ]

    # Use the time coordinate from the catalog dataset so the merged dataset is consistent.
    filtered_data = filtered_data.assign_coords(time=time_coord)

    return filtered_data.rename(var_name)

## 5. Replace the original variables with filtered variables

When `use_filtered_data` is `True`, the original variables are first removed from the base dataset. The filtered variables are then opened from Zarr and merged back into the OceanSpy `OceanDataset`.

In [6]:
def open_filtered_variable(var_name, filtered_data_root, time_coord):
    zarr_pattern = os.path.join(
        filtered_data_root,
        f"{var_name}_filt_each_time",
        f"{var_name}_filt_lowpass_20_time_*.zarr",
    )
    zarr_files = sorted(glob.glob(zarr_pattern))

    if not zarr_files:
        raise FileNotFoundError(
            f"No filtered Zarr files found for {var_name}: {zarr_pattern}"
        )

    print(f"{var_name}: opening {len(zarr_files)} Zarr stores")

    datasets = [
        xr.open_zarr(
            zarr_file,
            chunks={},              # preserve stored Zarr chunks
            consolidated=True,     # set True if metadata is consolidated
        )
        for zarr_file in zarr_files
    ]

    da = xr.concat(
        datasets,
        dim="time",
        data_vars="minimal",
        coords="minimal",
        compat="override",
        join="override",
    )[f"{var_name}_filt_lowpass_20"]

    da = da.assign_coords(time=time_coord)
    da = da.rename(var_name)

    return da

In [7]:
if use_filtered_data:
    ds_without_filtered_vars = od.dataset.drop_vars(filtered_var_names)

    time_coord = ds_without_filtered_vars.time

    filtered_arrays = []
    for var_name in filtered_var_names:
        filtered_arrays.append(
            open_filtered_variable(
                var_name=var_name,
                filtered_data_root=filtered_data_root,
                time_coord=time_coord,
            )
        )

    ds_filtered = xr.merge(
        filtered_arrays,
        compat="override",
        join="override",
    )

    ds_combined = xr.merge(
        [ds_without_filtered_vars, ds_filtered],
        compat="override",
        join="override",
    )

    od = ospy.OceanDataset(ds_combined)

ds = od.dataset

U: opening 1464 Zarr stores
V: opening 1464 Zarr stores
W: opening 1464 Zarr stores
S: opening 1464 Zarr stores
Temp: opening 1464 Zarr stores


## 6. Optional checks

The cells below provide simple checks that the expected variables are present and that the final dataset has a time coordinate.

In [8]:
print("Variables available in the final dataset:")
print([var for var in filtered_var_names if var in ds.data_vars])

print("\nDataset dimensions:")
print(ds.dims)

print("\nTime coordinate:")
print(ds.time)

Variables available in the final dataset:
['U', 'V', 'W', 'S', 'Temp']

Dataset dimensions:
FrozenMappingWarningOnValuesAccess({'Y': 880, 'X': 960, 'time': 1464, 'Z': 216, 'Yp1': 881, 'Xp1': 961, 'Zp1': 217, 'Zl': 216, 'Zu': 216, 'time_midp': 1463})

Time coordinate:
<xarray.DataArray 'time' (time: 1464)> Size: 12kB
array(['2007-09-01T00:00:00.000000000', '2007-09-01T06:00:00.000000000',
       '2007-09-01T12:00:00.000000000', ..., '2008-08-31T06:00:00.000000000',
       '2008-08-31T12:00:00.000000000', '2008-08-31T18:00:00.000000000'],
      shape=(1464,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 12kB 2007-09-01 ... 2008-08-31T18:00:00
Attributes:
    long_name:  model_time
